In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import numpy as np
import tensorflow as tf
import librosa
import pickle
import soundfile as sf
import matplotlib.pyplot as plt
from tqdm import tqdm
import scipy.signal as signal

2025-05-20 08:15:01.103113: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-20 08:15:01.229271: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747700101.278131  362291 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747700101.292563  362291 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1747700101.400031  362291 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [112]:
import numpy as np
import librosa
import soundfile as sf
import pickle
import tensorflow as tf
from scipy import signal
import os

# Define the dictionary structure based on the provided data
def create_emotion_data_dict():
    # Use the provided emotion data structure directly
    data = {
        'ANGRY': {
            'ZCR_mean': 0.0846,
            'ZCR_std': 0.0296,
            'RMSE_mean': 0.0692,
            'RMSE_std': 0.0478,
            'MFCC_means': [
                -298.124, 121.593, -7.995, 41.206, -17.941, 
                9.999, -15.744, 4.588, -15.002, 1.481,
                -3.137, -5.640, 3.305, -9.326, 2.893,
                -9.847, 0.812, -8.127, 0.116, -3.868
            ],
            'MFCC_stds': [
                57.716, 17.098, 15.712, 11.359, 9.764,
                10.442, 7.793, 5.566, 4.787, 4.461,
                4.062, 3.706, 3.975, 3.546, 3.853,
                4.285, 4.058, 3.810, 3.839, 3.803
            ]
        },
        'DISGUST': {
            'ZCR_mean': 0.0708,
            'ZCR_std': 0.0300,
            'RMSE_mean': 0.0212,
            'RMSE_std': 0.0174,
            'MFCC_means': [
                -385.598, 137.840, 1.608, 51.619, -15.655,
                22.205, -18.391, 9.706, -13.179, 3.508,
                -0.987, -4.419, 4.442, -9.774, 5.044,
                -10.782, 2.495, -8.586, 0.965, -4.649
            ],
            'MFCC_stds': [
                46.325, 14.257, 11.958, 12.195, 7.744,
                10.923, 6.812, 6.224, 4.110, 4.033,
                3.871, 3.139, 3.298, 2.954, 3.260,
                3.681, 3.200, 2.723, 2.941, 2.603
            ]
        },
        'FEAR': {
            'ZCR_mean': 0.0645,
            'ZCR_std': 0.0249,
            'RMSE_mean': 0.0303,
            'RMSE_std': 0.0310,
            'MFCC_means': [
                -379.109, 132.062, 4.086, 50.065, -14.141,
                21.857, -16.556, 8.178, -12.486, 2.971,
                -2.549, -5.420, 3.570, -9.721, 4.359,
                -10.538, 1.967, -8.359, 0.623, -4.244
            ],
            'MFCC_stds': [
                59.114, 15.984, 12.690, 12.302, 7.665,
                11.356, 6.379, 6.280, 3.946, 4.018,
                3.989, 3.531, 3.845, 3.502, 3.727,
                4.320, 3.925, 3.728, 3.632, 3.724
            ]
        },
        'HAPPY': {
            'ZCR_mean': 0.0671,
            'ZCR_std': 0.0255,
            'RMSE_mean': 0.0340,
            'RMSE_std': 0.0235,
            'MFCC_means': [
                -356.270, 133.676, 0.348, 45.186, -13.289,
                15.806, -16.239, 6.152, -13.946, 2.110,
                -2.470, -5.475, 3.416, -9.233, 4.003,
                -9.829, 1.275, -7.657, 0.534, -3.884
            ],
            'MFCC_stds': [
                47.268, 14.645, 13.520, 10.231, 8.411,
                10.349, 7.024, 5.542, 4.562, 4.244,
                4.177, 3.572, 3.609, 3.512, 3.479,
                4.405, 3.814, 3.650, 3.467, 3.484
            ]
        },
        'NEUTRAL': {
            'ZCR_mean': 0.0619,
            'ZCR_std': 0.0244,
            'RMSE_mean': 0.0167,
            'RMSE_std': 0.0062,
            'MFCC_means': [
                -398.677, 141.591, 6.849, 53.914, -14.457,
                21.870, -17.550, 8.230, -13.621, 3.048,
                -2.019, -4.408, 4.477, -9.200, 5.233,
                -10.508, 2.698, -8.454, 1.005, -4.425
            ],
            'MFCC_stds': [
                26.359, 12.097, 10.285, 8.730, 7.124,
                8.421, 6.028, 4.935, 3.831, 3.745,
                3.801, 2.985, 3.127, 2.819, 2.770,
                3.791, 2.989, 2.704, 2.824, 2.381
            ]
        },
        'SAD': {
            'ZCR_mean': 0.0553,
            'ZCR_std': 0.0218,
            'RMSE_mean': 0.0120,
            'RMSE_std': 0.0071,
            'MFCC_means': [
                -429.167, 143.558, 8.845, 58.525, -15.826,
                29.045, -17.954, 12.292, -12.491, 3.972,
                -1.606, -4.158, 4.409, -10.219, 5.771,
                -11.590, 3.432, -9.358, 1.490, -4.850
            ],
            'MFCC_stds': [
                37.168, 11.792, 9.642, 9.299, 6.888,
                9.068, 5.482, 5.340, 3.337, 3.396,
                3.323, 2.647, 2.934, 2.704, 2.720,
                3.494, 2.675, 2.329, 2.699, 2.500
            ]
        },
        'SURPRISE': {
            'ZCR_mean': 0.0952,
            'ZCR_std': 0.0534,
            'RMSE_mean': 0.0256,
            'RMSE_std': 0.0068,
            'MFCC_means': [
                -401.177, 86.208, 1.477, -3.929, -7.757,
                0.652, -7.570, -9.924, -12.696, -2.759,
                -15.193, -2.513, -6.028, 2.293, -2.100,
                2.053, 0.457, 5.550, -0.646, 1.538
            ],
            'MFCC_stds': [
                50.445, 25.650, 16.766, 15.021, 15.354,
                7.005, 6.511, 10.630, 6.926, 4.901,
                4.698, 6.992, 4.324, 5.505, 3.492,
                2.753, 4.170, 3.802, 3.333, 3.026
            ]
        }
    }
    return data

def load_saliency_curves(kde_dir="KDE_vals"):
    """
    Load the saliency curves for each emotion
    """
    saliency_data = {}
    for emotion in ['ANGRY', 'DISGUST', 'FEAR', 'HAPPY', 'NEUTRAL', 'SAD', 'SURPRISE']:
        try:
            sal_file = os.path.join(kde_dir, f"{emotion.lower()}.npz")
            if os.path.exists(sal_file):
                sal = np.load(sal_file)
                saliency_data[emotion] = {
                    'time': sal['time'],
                    'kde_scaled': sal['kde_scaled']
                }
            else:
                print(f"Warning: Saliency file not found for {emotion}")
                # Create default saliency curve (bell-shaped centered in the middle)
                time = np.linspace(0, 2.5, 100)
                kde_scaled = np.exp(-((time - 1.25) ** 2) / 0.5)
                saliency_data[emotion] = {
                    'time': time,
                    'kde_scaled': kde_scaled / np.max(kde_scaled)  # Normalize to [0, 1]
                }
        except Exception as e:
            print(f"Error loading saliency data for {emotion}: {e}")
            # Create default saliency curve
            time = np.linspace(0, 2.5, 100)
            kde_scaled = np.exp(-((time - 1.25) ** 2) / 0.5)
            saliency_data[emotion] = {
                'time': time,
                'kde_scaled': kde_scaled / np.max(kde_scaled)  # Normalize to [0, 1]
            }
    return saliency_data

# def generate_synthetic_audio(emotion, duration=2.5, sr=22050, data=None, saliency_data=None):
#     """
#     Generate synthetic audio with characteristics matching a specific emotion,
#     incorporating saliency curves to focus emotional characteristics where the model pays most attention
#     """
#     if data is None:
#         data = create_emotion_data_dict()
    
#     if saliency_data is None:
#         # Create default saliency curve (bell-shaped centered in the middle)
#         # Note: Emotion-specific saliency curves can be provided for better results
#         time = np.linspace(0, duration, 100)
#         kde_scaled = np.exp(-((time - duration/2) ** 2) / 0.5)
#         saliency = {
#             'time': time,
#             'kde_scaled': kde_scaled / np.max(kde_scaled)  # Normalize to [0, 1]
#         }
#     else:
#         # Use the provided saliency data for the specific emotion
#         if emotion in saliency_data:
#             saliency = saliency_data[emotion]
#         else:
#             # Default if the specific emotion saliency is not available
#             time = np.linspace(0, duration, 100)
#             kde_scaled = np.exp(-((time - duration/2) ** 2) / 0.5)
#             saliency = {
#                 'time': time,
#                 'kde_scaled': kde_scaled / np.max(kde_scaled)  # Normalize to [0, 1]
#             }
    
#     # Get the emotion data
#     emotion_data = data[emotion]
    
#     # Create a base audio signal
#     t = np.arange(0, duration, 1/sr)
    
#     # Interpolate saliency values to match the audio length
#     saliency_curve = np.interp(t, saliency['time'], saliency['kde_scaled'])
    
#     # Create a base signal with vocal tract resonance frequencies (formants)
#     f1, f2, f3 = 500, 1500, 2500  # First, second, and third formants
#     audio = 0.1 * np.sin(2 * np.pi * f1 * t) + 0.05 * np.sin(2 * np.pi * f2 * t) + 0.025 * np.sin(2 * np.pi * f3 * t)
    

#     # Apply RMSE-based amplitude modulation
#     rmse_mean, rmse_std = emotion_data['RMSE_mean'], emotion_data['RMSE_std']
#     frame_length, hop_length = 2048, 512
#     n_frames = int((sr*duration - frame_length)/hop_length) + 1
#     rmse_envelope = np.abs(np.random.normal(rmse_mean, rmse_std, n_frames))
#     rmse_curve = np.interp(t, np.linspace(0, duration, n_frames), rmse_envelope)
#     weighted_rmse = rmse_curve * (saliency_curve) * 5
#     audio *= weighted_rmse / np.mean(weighted_rmse)
    
#     zcr_mean = emotion_data['ZCR_mean']
#     zcr_std = emotion_data['ZCR_std']
#     noise_base = np.random.normal(zcr_mean, zcr_std, len(audio))
#     shaped_noise = noise_base * (0.2 + 0.8 * saliency_curve)
#     audio += shaped_noise
    
#     # Apply MFCC-based frequency components using mean and std
#     mfcc_means = emotion_data['MFCC_means']
#     mfcc_stds = emotion_data['MFCC_stds']
#     mfcc_contribution = np.zeros_like(audio)
#     for i in range(len(mfcc_means)):  # Limit to first 6 MFCCs
#         freq = 300 + i * 250
#         # Base amplitude from MFCC mean, with random variation from MFCC std
#         base_amp = np.abs(mfcc_means[i] / 500)
#         variation = np.random.normal(0, mfcc_stds[i] / 500, 1)[0]
#         amp = np.clip(base_amp + variation, 0, None)  # Ensure non-negative amplitude
#         mfcc_contribution += amp * np.sin(2 * np.pi * freq * t)
#     audio += mfcc_contribution * saliency_curve * 0.15

    
#     # Normalize audio to prevent clipping
#     audio = audio / np.max(np.abs(audio))
#     return audio
def generate_synthetic_audio(emotion, duration=2.5, sr=22050, data=None, saliency_data=None):
    """
    Generate synthetic audio based on RMSE, ZCR, and all 20 MFCC statistics for a specific emotion.
    
    Parameters:
    - emotion (str): The target emotion for audio generation.
    - duration (float): Duration of the audio in seconds (default: 2.5).
    - sr (int): Sample rate in Hz (default: 22050).
    - data (dict): Dictionary containing emotion-specific statistics (RMSE, ZCR, MFCC).
    - saliency_data (dict): Optional saliency data to emphasize emotional characteristics.
    
    Returns:
    - audio (np.ndarray): Generated audio signal.
    """
    if data is None:
        raise ValueError("Emotion data dictionary is required with RMSE, ZCR, and MFCC statistics.")
    if 'MFCC_means' not in data[emotion] or len(data[emotion]['MFCC_means']) < 20:
        raise ValueError("Emotion data must include at least 20 MFCC coefficients.")

    # Handle saliency data
    if saliency_data is None or emotion not in saliency_data:
        time = np.linspace(0, duration, 100)
        kde_scaled = np.exp(-((time - duration/2) ** 2) / 0.5)
        saliency = {
            'time': time,
            'kde_scaled': kde_scaled / np.max(kde_scaled)  # Normalize to [0, 1]
        }
    else:
        saliency = saliency_data[emotion]

    # Get emotion data
    emotion_data = data[emotion]
    t = np.arange(0, duration, 1/sr)

    # Interpolate saliency values to match audio length
    saliency_curve = np.interp(t, saliency['time'], saliency['kde_scaled'])

    if emotion.lower() == 'sad':
        base_freq = 200
    elif emotion.lower() == 'happy':
        base_freq = 500
    elif emotion.lower() == 'fear':
        base_freq = 300
    elif emotion.lower() == 'angry':
        base_freq = 1000
    elif emotion.lower() == 'neutral': #
        base_freq = (175 + 145)//2
    elif emotion.lower() == 'surprise': #
        base_freq = 0
    elif emotion.lower() == 'disgust': #
        base_freq = 0
    else:
        base_freq = 10

    audio = 0.1 * np.sin(2 * np.pi * base_freq * t)



    # Apply RMSE for amplitude modulation
    rmse_mean, rmse_std = emotion_data['RMSE_mean'], emotion_data['RMSE_std']
    frame_length, hop_length = 2048, 512
    n_frames = int((sr * duration - frame_length) / hop_length) + 1
    rmse_envelope = np.abs(np.random.normal(rmse_mean, rmse_std, n_frames))
    rmse_curve = np.interp(t, np.linspace(0, duration, n_frames), rmse_envelope)
    weighted_rmse = rmse_curve * saliency_curve
    audio *= weighted_rmse / np.mean(weighted_rmse)

    # Apply ZCR to add controlled noise
    zcr_mean, zcr_std = emotion_data['ZCR_mean'], emotion_data['ZCR_std']
    noise = np.random.normal(0, 1, len(audio))
    zcr_envelope = np.abs(np.random.normal(zcr_mean, zcr_std, n_frames))
    zcr_curve = np.interp(t, np.linspace(0, duration, n_frames), zcr_envelope)
    zcr_scaled = zcr_curve / np.max(zcr_curve) * 0.2
    audio += noise * zcr_scaled * saliency_curve

    # Apply all 20 MFCCs to shape frequency content
    mfcc_means = emotion_data['MFCC_means']
    mfcc_stds = emotion_data['MFCC_stds']
    mfcc_contribution = np.zeros_like(audio)
    for i in range(20):  # Use all 20 MFCCs
        # Spread frequencies from 300 Hz to 4300 Hz (avoid very high frequencies)
        freq = 300 + i * 200
        # Scale amplitude with a decay for higher MFCCs to reduce noise
        amp = np.abs(mfcc_means[i] / 500 + np.random.normal(0, mfcc_stds[i] / 500, 1)[0])
        amp = np.clip(amp, 0, None) * (1 - i * 0.03)  # Gradual decay for higher MFCCs
        mfcc_contribution += amp * np.sin(2 * np.pi * freq * t)
    audio += mfcc_contribution * saliency_curve * 0.08  # Reduced scaling for balance

    # Normalize audio to prevent clipping
    audio = audio / np.max(np.abs(audio))
    return audio

def create_all_emotion_audio(output_dir="emotion_audio", sr=22050):
    """Create audio files for all emotions"""
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    data = create_emotion_data_dict()
    saliency_data = load_saliency_curves()
    for emotion in data.keys():
        print(f"Generating {emotion} audio...")
        audio = generate_synthetic_audio(emotion, duration=2.5, sr=sr, data=data, saliency_data=saliency_data)
        sf.write(os.path.join(output_dir, f"{emotion.lower()}_synthetic.wav"), audio, sr)
        print(f"Saved to {output_dir}/{emotion.lower()}_synthetic.wav")

def extract_features(data, sr=22050, frame_length=2048, hop_length=512):
    """Extract ZCR, RMSE, and MFCC features from audio"""
    zcr = librosa.feature.zero_crossing_rate(data, frame_length=frame_length, hop_length=hop_length)
    rmse = librosa.feature.rms(y=data, frame_length=frame_length, hop_length=hop_length)
    mfcc_features = librosa.feature.mfcc(y=data, sr=sr, n_mfcc=20, n_fft=frame_length, hop_length=hop_length).T
    return np.hstack((np.squeeze(zcr), np.squeeze(rmse), np.ravel(mfcc_features)))

def load_model_and_predictors(model_json_path, model_weights_path, scaler_path, encoder_path):
    """Load model, scaler, and encoder with error handling"""
    try:
        with open(model_json_path, 'r') as json_file:
            model_json = json_file.read()
        model = tf.keras.models.model_from_json(model_json)
        model.load_weights(model_weights_path)
        model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
        with open(scaler_path, "rb") as f:
            scaler = pickle.load(f)
        with open(encoder_path, "rb") as f:
#             print(encoder_path)
            encoder = pickle.load(f)
        return model, scaler, encoder
    except FileNotFoundError as e:
        print(f"Error: File not found - {e}")
        return None, None, None
    except Exception as e:
        print(f"Error loading model or predictors: {e}")
        return None, None, None

def get_predict_feat(audio_data, sr, scaler, expected_shape=(1, 2376)):
    """Prepare features for prediction"""
    res = extract_features(audio_data, sr)
    flat_size = np.prod(expected_shape)
    if res.size < flat_size:
        res = np.pad(res, (0, flat_size - res.size), mode='constant')
    else:
        res = np.resize(res, expected_shape)
    return np.expand_dims(scaler.transform(res.reshape(1, -1)), axis=2)

def prediction(audio_data, sr, model, scaler, encoder):
    """Predict emotion from audio"""
    res = get_predict_feat(audio_data, sr, scaler)
    predictions = model.predict(res)
    label_names = list(encoder.categories_[0])
    predicted_label_index = np.argmax(predictions)
    confidence_scores = [{'label': label_name, 'confidence': max(0, predictions[0][i]) if predictions[0][i] >= 0.001 else 0}
                         for i, label_name in enumerate(label_names)]
    print(f"\nPredicted Emotion: {label_names[predicted_label_index]}")
    return sorted(confidence_scores, key=lambda x: x['confidence'], reverse=True)

def optimize_emotion_audio(emotion, model, scaler, encoder, iterations=20, sr=22050):
    """Optimize audio generation for target emotion"""
    best_audio, best_confidence = None, 0
    highest_audio, highest_confidence, highest_emo = None, 0, None
    
    data = create_emotion_data_dict()
    saliency_data = load_saliency_curves()
    
    for i in range(iterations):
        print(f"Iteration {i+1}/{iterations}")
        audio = generate_synthetic_audio(emotion, duration=2.5, sr=sr, data=data, saliency_data=saliency_data)
        results = prediction(audio, sr, model, scaler, encoder)
        target_result = next((r for r in results if r['label'] == emotion.lower()), None)
        print(target_result)
        if target_result and target_result['confidence'] > best_confidence:
            best_confidence = target_result['confidence']
            best_audio = audio
            print(f"New best confidence for {emotion}: {best_confidence:.4f}")
        
        highest_pred = max(results, key=lambda x: x['confidence'])
        if highest_pred['confidence'] > highest_confidence:
            highest_audio = audio
            highest_emo = highest_pred['label']
            highest_confidence = highest_pred['confidence']
    
    if best_audio is not None:
        print(f"Best confidence achieved for {emotion}: {best_confidence:.4f}")
        return best_audio
    print(f"Could not optimize audio for {emotion}")
    print("Predicted Emotion: ",  highest_emo)
    print("Confidence: ", highest_confidence)
    return highest_audio


if __name__ == "__main__":
#     create_all_emotion_audio()
#     Example optimization (uncomment and provide model paths if needed):
    model, scaler, encoder = load_model_and_predictors("results/CNN_model.json", "results/best_model.weights.h5", "results/scaler.pickle", "results/encoder.pickle")
    for emo in ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']:
        optimized_audio = optimize_emotion_audio(f"{emo.upper()}", model, scaler, encoder, 100)
        sf.write(f"try_optimized_synth/{emo}_optimized.wav", optimized_audio, 22050)

/home/aegis/Research/Machine Learning in HEP/mlenv/lib/python3.12/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.2.2 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/aegis/Research/Machine Learning in HEP/mlenv/lib/python3.12/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.2.2 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Iteration 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 206ms/step

Predicted Emotion: angry
{'label': 'angry', 'confidence': 0.99903166}
New best confidence for ANGRY: 0.9990
Iteration 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step

Predicted Emotion: angry
{'label': 'angry', 'confidence': 0.97581667}
Iteration 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step

Predicted Emotion: angry
{'label': 'angry', 'confidence': 0.7480452}
Iteration 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step

Predicted Emotion: angry
{'label': 'angry', 'confidence': 0.9493976}
Iteration 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step

Predicted Emotion: angry
{'label': 'angry', 'confidence': 0.95882595}
Iteration 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step

Predicted Emotion: angry
{'label': 'angry', 'confidence': 0.99851924}
Iteration 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step

Predicted Emotion: angry
{'label': 'angry', 'confidence': 0.99489516}
Iteration 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step

Predicted Emotion: angry
{'lab

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step

Predicted Emotion: angry
{'label': 'angry', 'confidence': 0.8783123}
Iteration 53/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step

Predicted Emotion: angry
{'label': 'angry', 'confidence': 0.999592}
Iteration 54/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step

Predicted Emotion: angry
{'label': 'angry', 'confidence': 0.9062169}
Iteration 55/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step

Predicted Emotion: angry
{'label': 'angry', 'confidence': 0.995214}
Iteration 56/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step

Predicted Emotion: angry
{'label': 'angry', 'confidence': 0.9937894}
Iteration 57/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step

Predicted Emotion: angry
{'label': 'angry', 'confidence': 0.9999366}
New best confidence for ANGRY: 0.9999
Iteration 58/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step

Predicted Emotion: angry
{'label': 'angry', 'confidence': 0.9793114}
Iteration 59/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step

Predicted Emotion: angry
{'label': 'angry', 'c

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step

Predicted Emotion: angry
{'label': 'disgust', 'confidence': 0.0016505007}
Iteration 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step

Predicted Emotion: angry
{'label': 'disgust', 'confidence': 0.024502851}
Iteration 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step

Predicted Emotion: angry
{'label': 'disgust', 'confidence': 0.0050530406}
Iteration 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step

Predicted Emotion: angry
{'label': 'disgust', 'confidence': 0}
Iteration 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step

Predicted Emotion: angry
{'label': 'disgust', 'confidence': 0.008835843}
Iteration 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step

Predicted Emotion: angry
{'label': 'disgust', 'confidence': 0}
Iteration 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step

Predicted Emotion: angry
{'label': 'disgust', 'confidence': 0}
Iteration 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step

Predicted Emotion: angry
{'label': 'disgust', 'confidence': 0}
Iteration 10/100
1/1 ━━━

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step

Predicted Emotion: angry
{'label': 'disgust', 'confidence': 0.02870692}
Iteration 54/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step

Predicted Emotion: angry
{'label': 'disgust', 'confidence': 0.05345891}
Iteration 55/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step

Predicted Emotion: angry
{'label': 'disgust', 'confidence': 0}
Iteration 56/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step

Predicted Emotion: angry
{'label': 'disgust', 'confidence': 0}
Iteration 57/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step

Predicted Emotion: angry
{'label': 'disgust', 'confidence': 0.0038306848}
Iteration 58/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step

Predicted Emotion: angry
{'label': 'disgust', 'confidence': 0.005998955}
Iteration 59/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step

Predicted Emotion: angry
{'label': 'disgust', 'confidence': 0}
Iteration 60/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step

Predicted Emotion: angry
{'label': 'disgust', 'confidence': 0.07872674}
Iteration 61/1

...

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step

Predicted Emotion: sad
{'label': 'neutral', 'confidence': 0}
Iteration 64/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step

Predicted Emotion: sad
{'label': 'neutral', 'confidence': 0}
Iteration 65/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step

Predicted Emotion: sad
{'label': 'neutral', 'confidence': 0}
Iteration 66/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step

Predicted Emotion: sad
{'label': 'neutral', 'confidence': 0}
Iteration 67/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step

Predicted Emotion: sad
{'label': 'neutral', 'confidence': 0}
Iteration 68/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step

Predicted Emotion: sad
{'label': 'neutral', 'confidence': 0}
Iteration 69/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step

Predicted Emotion: sad
{'label': 'neutral', 'confidence': 0}
Iteration 70/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step

Predicted Emotion: sad
{'label': 'neutral', 'confidence': 0}
Iteration 71/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step

Predicted Emotion: sa

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step

Predicted Emotion: sad
{'label': 'sad', 'confidence': 0.99940944}
Iteration 17/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step

Predicted Emotion: sad
{'label': 'sad', 'confidence': 0.9995542}
Iteration 18/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step

Predicted Emotion: sad
{'label': 'sad', 'confidence': 0.999746}
Iteration 19/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step

Predicted Emotion: sad
{'label': 'sad', 'confidence': 0.9998273}
Iteration 20/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step

Predicted Emotion: sad
{'label': 'sad', 'confidence': 0.9999201}
New best confidence for SAD: 0.9999
Iteration 21/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step

Predicted Emotion: sad
{'label': 'sad', 'confidence': 0.9995197}
Iteration 22/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step

Predicted Emotion: sad
{'label': 'sad', 'confidence': 0.9999598}
New best confidence for SAD: 1.0000
Iteration 23/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step

Predicted Emotion: sad
{'label': 'sad', 

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step

Predicted Emotion: sad
{'label': 'sad', 'confidence': 0.9998735}
Iteration 69/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step

Predicted Emotion: sad
{'label': 'sad', 'confidence': 0.9998634}
Iteration 70/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step

Predicted Emotion: sad
{'label': 'sad', 'confidence': 0.99983764}
Iteration 71/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step

Predicted Emotion: sad
{'label': 'sad', 'confidence': 0.9998066}
Iteration 72/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step

Predicted Emotion: sad
{'label': 'sad', 'confidence': 0.99980634}
Iteration 73/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step

Predicted Emotion: sad
{'label': 'sad', 'confidence': 0.9998072}
Iteration 74/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step

Predicted Emotion: sad
{'label': 'sad', 'confidence': 0.99966455}
Iteration 75/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step

Predicted Emotion: sad
{'label': 'sad', 'confidence': 0.99987173}
Iteration 76/100
1/1 ━━━━━━━━━━━━━━━━━━━

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step

Predicted Emotion: angry
{'label': 'surprise', 'confidence': 0}
Iteration 21/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step

Predicted Emotion: fear
{'label': 'surprise', 'confidence': 0.002454414}
Iteration 22/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step

Predicted Emotion: angry
{'label': 'surprise', 'confidence': 0}
Iteration 23/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step

Predicted Emotion: fear
{'label': 'surprise', 'confidence': 0.019436304}
Iteration 24/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step

Predicted Emotion: fear
{'label': 'surprise', 'confidence': 0}
Iteration 25/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step

Predicted Emotion: angry
{'label': 'surprise', 'confidence': 0}
Iteration 26/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step

Predicted Emotion: happy
{'label': 'surprise', 'confidence': 0}
Iteration 27/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step

Predicted Emotion: fear
{'label': 'surprise', 'confidence': 0}
Iteration 28/100
1/1 ━━━━━━━━━━━━━━━━━━

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step

Predicted Emotion: angry
{'label': 'surprise', 'confidence': 0}
Iteration 73/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step

Predicted Emotion: angry
{'label': 'surprise', 'confidence': 0}
Iteration 74/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step

Predicted Emotion: angry
{'label': 'surprise', 'confidence': 0.0019615279}
Iteration 75/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step

Predicted Emotion: angry
{'label': 'surprise', 'confidence': 0.0014504518}
Iteration 76/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step

Predicted Emotion: disgust
{'label': 'surprise', 'confidence': 0}
Iteration 77/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step

Predicted Emotion: angry
{'label': 'surprise', 'confidence': 0}
Iteration 78/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step

Predicted Emotion: angry
{'label': 'surprise', 'confidence': 0}
Iteration 79/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step

Predicted Emotion: fear
{'label': 'surprise', 'confidence': 0}
Iteration 80/100
1/1 ━━━━━━━━━

In [114]:
for file in os.listdir("try_optimized_synth"):
    filepath = os.path.join("try_optimized_synth", file)
    prediction(filepath)

try_optimized_synth/neutral_synthetic.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 200ms/step

Predicted Emotion: neutral


try_optimized_synth/fear_optimized.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step

Predicted Emotion: sad


try_optimized_synth/happy_optimized.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step

Predicted Emotion: angry


try_optimized_synth/angry_optimized.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step

Predicted Emotion: sad


try_optimized_synth/neutral_optimized.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step

Predicted Emotion: sad


try_optimized_synth/sad_optimized.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step

Predicted Emotion: sad


try_optimized_synth/surprise_optimized.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step

Predicted Emotion: sad


try_optimized_synth/disgust_optimized.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step

Predicted Emotion: disgust


